In [3]:
!pip install nlpaug transformers==4.38.2 datasets pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.16.1
    Uninstalling huggingface_hub-1.16.1:
      Successfully uninstalled huggingface_hub-1.16.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into acc

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import nlpaug.augmenter.word as naw
from datasets import load_dataset
import os
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

print("1. Aşama: Veri seti indiriliyor...")
dataset = load_dataset("Overfit-GM/turkish-toxic-language")
df_original = dataset['train'].to_pandas()

print("2. Aşama: BERTurk modeli GPU'ya yükleniyor...")
aug = naw.ContextualWordEmbsAug(
    model_path='dbmdz/bert-base-turkish-cased',
    action="substitute",
    aug_p=0.3,
    device='cuda'
)

# DOSYAYI DOĞRUDAN GOOGLE DRIVE'A KAYDEDİYORUZ
output_file = "/content/drive/MyDrive/1_milyon_toksik_veri.csv"

def generate_synthetic_data(df, samples_per_row=12, batch_size=5000, output_file=output_file):

    start_index = 0

    # KALDIĞI YERDEN DEVAM ETME MANTIĞI (RESUME)
    if os.path.exists(output_file):
        try:
            # Mevcut dosyayı oku ve kaç orijinal satırın işlendiğini hesapla
            mevcut_df = pd.read_csv(output_file)
            islenen_satir = len(mevcut_df) // samples_per_row
            start_index = islenen_satir
            print(f"-> DİKKAT: Dosya zaten var! İşleme baştan değil, {start_index}. satırdan devam edilecek.\n")
        except:
            pass
    else:
        # Dosya yoksa sıfırdan başlıklarla oluştur
        pd.DataFrame(columns=['text', 'is_toxic', 'is_synthetic']).to_csv(output_file, index=False)
        print("-> Yeni dosya oluşturuldu. İşlem sıfırdan başlıyor.\n")

    synthetic_rows = []

    print("3. Aşama: Sentetik veri üretimi başlıyor...")

    # Sadece işlenmemiş satırları al
    df_to_process = df.iloc[start_index:]

    for index, row in tqdm(df_to_process.iterrows(), total=df_to_process.shape[0]):
        original_text = row['text']
        toxic_label = row['is_toxic']

        try:
            augmented_texts = aug.augment(original_text, n=samples_per_row)
            for aug_text in augmented_texts:
                synthetic_rows.append({
                    'text': aug_text,
                    'is_toxic': toxic_label,
                    'is_synthetic': 1
                })
        except Exception as e:
            continue

        # Drive'a parça parça kaydet
        if len(synthetic_rows) >= batch_size:
            df_batch = pd.DataFrame(synthetic_rows)
            df_batch.to_csv(output_file, mode='a', header=False, index=False)
            synthetic_rows = []

    if len(synthetic_rows) > 0:
        df_batch = pd.DataFrame(synthetic_rows)
        df_batch.to_csv(output_file, mode='a', header=False, index=False)

    print(f"\n-> İŞLEM TAMAMLANDI! Veriler Drive'ına kaydedildi.")

# Üretimi Başlat
generate_synthetic_data(df=df_original, samples_per_row=12, batch_size=5000)

1. Aşama: Veri seti indiriliyor...


README.md: 0.00B [00:00, ?B/s]

turkish_toxic_language.csv:   0%|          | 0.00/18.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/77800 [00:00<?, ? examples/s]

2. Aşama: BERTurk modeli GPU'ya yükleniyor...


...

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

-> DİKKAT: Dosya zaten var! İşleme baştan değil, 17931. satırdan devam edilecek.

3. Aşama: Sentetik veri üretimi başlıyor...


  5%|▌         | 3008/59869 [58:49<19:12:34,  1.22s/it]

In [ ]:
print(df_original.columns)

Index(['text', 'target', 'source', 'is_toxic'], dtype='object')
